In [1]:
# Install dependencies for Unsloth + GPT-OSS
!pip install --upgrade -qqq uv
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" numpy pillow torchvision bitsandbytes "transformers==4.56.2" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Install openai_harmony (Harmony protocol tools)
!pip install -q openai-harmony jupyter_client pandas datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 117.6 MB/s eta 0:00:0000:0100:01
Using Python 3.12.12 environment at: /usr
Resolved 5 packages in 39ms                                          
Prepared 1 package in 64ms                                               
Uninstalled 1 package in 1ms
Installed 1 package in 6ms                                  
 - trl==0.24.0
 + trl==0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 50.3 MB/s eta 0:00:00a 0:00:01


# Load GPT-OSS 20B Model with Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch

# Load GPT-OSS 20B model with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-20b",
    max_seq_length=32768,  # Adjust based on your GPU memory
    load_in_4bit=False,    # Set True for lower VRAM usage
)

# Enable faster inference
FastLanguageModel.for_inference(model)

print(f"Model loaded on device: {model.device}")
print(f"Model dtype: {model.dtype}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:2525: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.9: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00000-of-00002.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Model loaded on device: cuda:0
Model dtype: torch.bfloat16


In [3]:
%%writefile local_python_tool.py
"""Python tool using Jupyter kernel for stateful execution."""
import os
import queue
import threading
from abc import ABC, abstractmethod
from typing import AsyncIterator, Any
from uuid import UUID, uuid4

from openai_harmony import (
    Author,
    Content,
    Message,
    Role,
    TextContent,
    ToolNamespaceConfig,
)


def add_libs(code: str) -> str:
    """Add common math libraries to code."""
    return "import math\nimport numpy as np\nimport sympy as sp\nfrom sympy import *\n" + code


def ensure_last_print(code: str) -> str:
    """Ensure the last expression is printed."""
    lines = code.strip().split("\n")
    if lines and "print(" not in lines[-1] and "import" not in lines[-1]:
        if "#" in lines[-1]:
            lines[-1] = lines[-1].split("#")[0]
        lines[-1] = "print(" + lines[-1] + ")"
    return "\n".join(lines)


class LocalJupyterSession:
    """Stateful Jupyter kernel session for code execution."""

    # Class-level lock and port counter to avoid port conflicts
    _port_lock = threading.Lock()
    _next_port = 50000
    _max_port = 65535  # Maximum valid port number

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:
        """Get next available ports for kernel connection."""
        import socket
        with cls._port_lock:
            ports = []
            attempts = 0
            max_attempts = 100  # Prevent infinite loop
            
            while len(ports) < count and attempts < max_attempts:
                start_port = cls._next_port
                # Check if port range is available
                available = True
                for i in range(count):
                    port = start_port + i
                    if port > cls._max_port:
                        # Wrap around to beginning of port range
                        start_port = 50000
                        port = start_port + i
                    
                    # Quick check if port is in use
                    try:
                        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                            s.settimeout(0.1)
                            result = s.connect_ex(('127.0.0.1', port))
                            if result == 0:
                                available = False
                                break
                    except Exception:
                        # If check fails, assume port might be in use
                        available = False
                        break
                
                if available:
                    ports = list(range(start_port, start_port + count))
                    cls._next_port = start_port + count
                    if cls._next_port > cls._max_port:
                        cls._next_port = 50000
                    break
                else:
                    # Try next range
                    cls._next_port += count
                    if cls._next_port > cls._max_port:
                        cls._next_port = 50000
                    attempts += 1
            
            if len(ports) < count:
                # Fallback: just return sequential ports without checking
                ports = list(range(cls._next_port, cls._next_port + count))
                cls._next_port += count
                if cls._next_port > cls._max_port:
                    cls._next_port = 50000
            
            return ports

    def __init__(self, connection_file: str | None = None, *, timeout: float = 120.0):
        try:
            from jupyter_client import BlockingKernelClient, KernelManager
        except ImportError as exc:
            raise RuntimeError("jupyter_client package required") from exc

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client: BlockingKernelClient
        self._km: KernelManager | None = None

        if connection_file:
            from pathlib import Path
            connection_path = Path(connection_file).expanduser()
            if not connection_path.exists():
                raise FileNotFoundError(f"Connection file not found: {connection_path}")
            client = BlockingKernelClient()
            client.load_connection_file(str(connection_path))
            client.start_channels()
            client.wait_for_ready(timeout=self._default_timeout)
            self._client = client
        else:
            # Allocate unique ports to avoid conflicts when running multiple kernels
            ports = self._get_next_ports(5)
            km = None
            max_retries = 3
            for retry in range(max_retries):
                try:
                    km = KernelManager()
                    km.shell_port = ports[0]
                    km.iopub_port = ports[1]
                    km.stdin_port = ports[2]
                    km.hb_port = ports[3]
                    km.control_port = ports[4]
                    km.start_kernel()
                    client = km.blocking_client()
                    client.start_channels()
                    client.wait_for_ready(timeout=self._default_timeout)
                    self._client = client
                    self._km = km
                    self._owns_kernel = True
                    break
                except Exception as e:
                    if retry < max_retries - 1:
                        # Try different ports
                        ports = self._get_next_ports(5)
                        if km is not None:
                            try:
                                km.shutdown_kernel(now=True)
                            except Exception:
                                pass
                    else:
                        # Last retry failed, raise the exception
                        raise RuntimeError(f"Failed to start kernel after {max_retries} retries: {e}") from e

    def execute(self, code: str, *, timeout: float | None = None) -> str:
        """Execute code and return combined stdout/stderr.
        timeout: WALL-CLOCK seconds limit for this execution.
        """
        import time
        import queue as _queue
    
        client = self._client
        effective_timeout = float(timeout or self._default_timeout)
    
        msg_id = client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
    
        stdout_parts: list[str] = []
        stderr_parts: list[str] = []
        
        # Track if we've seen a timeout/interrupt to filter IPython internal errors
        _timeout_triggered = False
    
        start = time.time()
        poll = 0.5  # seconds: small polling interval so we can enforce wall-clock timeout
    
        def _timed_out() -> bool:
            return (time.time() - start) >= effective_timeout
    
        # iopub loop
        max_timeout_grace = 1.0  # Give kernel 1 seconds to clean up after interrupt
        timeout_grace_start = None
        
        while True:
            if _timed_out():
                if not _timeout_triggered:
                    _timeout_triggered = True
                    timeout_grace_start = time.time()
                    # interrupt the kernel to stop runaway execution
                    try:
                        # BlockingKernelClient usually has interrupt_kernel
                        client.interrupt_kernel()
                    except Exception:
                        try:
                            if self._owns_kernel and self._km is not None:
                                self._km.interrupt_kernel()
                        except Exception:
                            pass
                
                # After grace period, stop collecting messages and raise timeout
                if timeout_grace_start and (time.time() - timeout_grace_start) > max_timeout_grace:
                    raise TimeoutError(f"Python execution exceeded wall-time limit: {effective_timeout:.1f}s")
    
            try:
                msg = client.get_iopub_msg(timeout=poll)
            except _queue.Empty:
                if _timeout_triggered and timeout_grace_start and (time.time() - timeout_grace_start) > max_timeout_grace:
                    raise TimeoutError(f"Python execution exceeded wall-time limit: {effective_timeout:.1f}s")
                continue
    
            if msg.get("parent_header", {}).get("msg_id") != msg_id:
                continue
    
            msg_type = msg.get("msg_type")
            content = msg.get("content", {})
            
            # After timeout is triggered, only collect essential messages and filter IPython errors
            if _timeout_triggered:
                # Only process status messages to detect idle state, ignore everything else
                if msg_type == "status":
                    if content.get("execution_state") == "idle":
                        break
                # Skip all other messages after timeout to avoid IPython internal errors
                continue
    
            if msg_type == "stream":
                text = content.get("text", "")
                if content.get("name") == "stdout":
                    stdout_parts.append(text)
                else:
                    stderr_parts.append(text)
            elif msg_type == "error":
                traceback_data = content.get("traceback")
                if traceback_data:
                    stderr_parts.append("\n".join(traceback_data))
                else:
                    ename = content.get("ename", "")
                    evalue = content.get("evalue", "")
                    stderr_parts.append(f"{ename}: {evalue}".strip())
            elif msg_type in {"execute_result", "display_data"}:
                data = content.get("data", {})
                text = data.get("text/plain")
                if text:
                    stdout_parts.append(text if text.endswith("\n") else f"{text}\n")
            elif msg_type == "status" and content.get("execution_state") == "idle":
                break
    
        # shell reply (also wall-time protected)
        # Reuse timeout_grace_start from iopub loop if timeout was already triggered
        shell_timeout_grace_start = timeout_grace_start if _timeout_triggered else None
        
        while True:
            if _timed_out():
                if not _timeout_triggered:
                    _timeout_triggered = True
                    shell_timeout_grace_start = time.time()
                    try:
                        client.interrupt_kernel()
                    except Exception:
                        try:
                            if self._owns_kernel and self._km is not None:
                                self._km.interrupt_kernel()
                        except Exception:
                            pass
                
                # After grace period, stop collecting messages and raise timeout
                if shell_timeout_grace_start and (time.time() - shell_timeout_grace_start) > max_timeout_grace:
                    raise TimeoutError(f"Python execution exceeded wall-time limit: {effective_timeout:.1f}s")
    
            try:
                reply = client.get_shell_msg(timeout=poll)
            except _queue.Empty:
                if _timeout_triggered and shell_timeout_grace_start and (time.time() - shell_timeout_grace_start) > max_timeout_grace:
                    raise TimeoutError(f"Python execution exceeded wall-time limit: {effective_timeout:.1f}s")
                continue
    
            if reply.get("parent_header", {}).get("msg_id") != msg_id:
                continue
    
            reply_content = reply.get("content", {})
            
            # After timeout, skip error messages to avoid IPython internal errors
            if _timeout_triggered and reply_content.get("status") == "error":
                # Skip IPython internal errors, just break to exit
                break
            
            if reply_content.get("status") == "error":
                traceback_data = reply_content.get("traceback")
                if traceback_data:
                    stderr_parts.append("\n".join(traceback_data))
                else:
                    ename = reply_content.get("ename", "")
                    evalue = reply_content.get("evalue", "")
                    stderr_parts.append(f"{ename}: {evalue}".strip())
            break
    
        stdout = "".join(stdout_parts)
        stderr = "".join(stderr_parts)
    
        if stderr:
            stdout = f"{stdout.rstrip()}\n{stderr}" if stdout else stderr
        if not stdout.strip():
            stdout = "[WARN] No output. Use print() to see results."
        return stdout


    def close(self):
        import contextlib
        with contextlib.suppress(Exception):
            self._client.stop_channels()
        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

    def __del__(self):
        self.close()


class PythonTool:
    """Python execution tool using Jupyter kernel."""

    def __init__(self, execution_backend: str | None = None, local_jupyter_timeout: float = 60.0):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._execution_lock = threading.Lock()
        self._jupyter_session: LocalJupyterSession | None = None
        # Lazy initialization to avoid port conflicts during object creation
        self._init_lock = threading.Lock()

    def _ensure_session(self):
        """Lazily initialize the Jupyter session."""
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = LocalJupyterSession(timeout=self._local_jupyter_timeout)

    @classmethod
    def get_tool_name(cls) -> str:
        return "python"

    @property
    def name(self) -> str:
        return self.get_tool_name()

    @property
    def instruction(self) -> str:
        return """Use this tool to execute Python code. The code runs in a stateful Jupyter notebook. Use print() to see output."""

    @property
    def tool_config(self) -> ToolNamespaceConfig:
        return ToolNamespaceConfig(
            name=self.get_tool_name(),
            description=self.instruction,
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:
        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name=self.get_tool_name())
        message = Message(author=author, content=[content]).with_recipient("assistant")
        if channel:
            message = message.with_channel(channel)
        return message

    def process_sync_plus(self, message: Message, timeout: float | None = None) -> list[Message]:
        """Execute code from message using Jupyter kernel."""
        self._ensure_session()
        script = message.content[0].text
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(script, timeout=timeout)
            except TimeoutError as exc:
                output = f"[ERROR] {exc}"
            except Exception as exc:
                output = f"[ERROR] {exc}"
        return [self._make_response(output, channel=message.channel)]

    def close(self):
        if self._jupyter_session is not None:
            self._jupyter_session.close()
            self._jupyter_session = None

    def __del__(self):
        self.close()

Writing local_python_tool.py


# Imports and Setup

In [4]:
import warnings
warnings.simplefilter('ignore')

import time
import re
import math
import threading
import queue
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List

import torch
import pandas as pd
from transformers import set_seed, TextStreamer

from openai_harmony import (
    HarmonyEncodingName,
    load_harmony_encoding,
    Conversation,
    Message,
    Role,
    SystemContent,
    ReasoningEffort,
    RenderConversationConfig,
)

from local_python_tool import PythonTool

# Load Harmony encoding for GPT-OSS
encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)

# Constants
SEED = 42
set_seed(SEED)
MAX_LEN = 32768
USE_BUDGET = False
K = 4  # Number of parallel samples (reduced for single GPU with Unsloth)

# Inference parameters
TEMPERATURE = 1.0
TOP_P = 1.0
MIN_P = 0.02

# TIR Prompts

In [5]:
# TIR (Tool-Integrated Reasoning) Prompt
TIR_PROMPTS = [
    """Please reason step by step and use the python tool to solve the math problem.
Finally, Return only the verified final answer in \\boxed{}, where the answer is an integer. Never guess."""
]

# Inferencer with Harmony Protocol

In [6]:
# Create Python tool pool for code execution
python_pool = queue.Queue(maxsize=K)

for _ in range(K):
    t = PythonTool(execution_backend="jupyter", local_jupyter_timeout=60.0)
    python_pool.put(t)
print(f"Python tool pool created with {K} instances!")

Python tool pool created with 4 instances!


In [7]:
class UnslothTIRInferencer:
    """Inferencer using Unsloth with Harmony protocol and Tool-Integrated Reasoning (TIR)."""

    def __init__(
        self,
        model,
        tokenizer,
        max_model_len: int = MAX_LEN,
        temperature: float = TEMPERATURE,
        top_p: float = TOP_P,
        min_p: float = MIN_P,
        seed: int = SEED,
        k: int = K,
        use_budget: bool = USE_BUDGET,
        max_iter: int = 50,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.max_model_len = max_model_len
        self.temperature = temperature
        self.top_p = top_p
        self.min_p = min_p
        self.seed = seed
        self.k = k
        self.use_budget = use_budget
        self.max_iter = max_iter
        self.deadline = None
        self.device = getattr(model, "device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        
        # Get stop token ids from Harmony encoding
        self.stop_token_ids = encoding.stop_tokens_for_assistant_actions()

    def get_num_samples(self) -> int:
        """Determine number of samples to generate."""
        print(f"Samples: {self.k}")
        return self.k
            
    def apply_chat_template(self, prompt: str, python_tool: PythonTool) -> list[Message]:
        """Wrap user prompt into Harmony conversation format with system and tool info."""
        return [
            Message.from_role_and_content(
                Role.SYSTEM,
                SystemContent.new()
                .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
                .with_tools(python_tool.tool_config)
            ),
            Message.from_role_and_content(Role.USER, prompt),
        ]

    def format_prompts(self, problem: str) -> list[str]:
        """Create multiple prompts for one problem."""
        num_samples = self.get_num_samples()
        prompts = []
        for i in range(num_samples):
            tir_prompt = TIR_PROMPTS[i % len(TIR_PROMPTS)]
            prompts.append(problem + "\n\n" + tir_prompt)
        return prompts

    def inference(self, problem: str, deadline: float = None) -> tuple[int, float]:
        """Run multi-sample inference for a single problem."""
        if deadline is None:
            deadline = time.time() + 600  # 10 min default
        self.deadline = deadline
        start_time = time.time()
    
        prompts = self.format_prompts(problem)
        responses = self._inference_sequential(prompts)
    
        duration = time.time() - start_time
        saved_time = max(0.0, deadline - time.time())
    
        print(f"[inference] Took {duration:.2f}s")
    
        return self.parse_responses(responses), saved_time

    def _generate_with_unsloth(self, input_ids: torch.Tensor, max_new_tokens: int = 2048) -> torch.Tensor:
        """Generate tokens using Unsloth model."""
        # Ensure input is on device
        if hasattr(input_ids, "to"):
            input_ids = input_ids.to(self.device)
        
        # Build generation config
        gen_kwargs = dict(
            max_new_tokens=max_new_tokens,
            min_new_tokens=16,
            do_sample=self.temperature > 0,
            temperature=self.temperature if self.temperature > 0 else None,
            top_p=self.top_p if self.temperature > 0 else None,
            pad_token_id=self.tokenizer.eos_token_id if self.tokenizer.pad_token_id is None else self.tokenizer.pad_token_id,
            eos_token_id=self.stop_token_ids,
        )
        
        with torch.no_grad():
            outputs = self.model.generate(
                input_ids=input_ids,
                **gen_kwargs
            )
        
        return outputs

    def single_generate_tir(self, prompt: str, seed_offset: int = 0) -> str:
        """Generate single TIR response with tool execution using Unsloth."""
        python_tool = None
    
        def _compute_py_timeout() -> float:
            PY_CUSHION = 1.0
            MAX_PY_TIMEOUT = 15.0
            MIN_ALLOW = 0.2
    
            if not getattr(self, "deadline", None):
                return MAX_PY_TIMEOUT
    
            remaining = self.deadline - time.time()
            t = remaining - PY_CUSHION
            if t <= 0:
                return 0.0
    
            return min(MAX_PY_TIMEOUT, max(MIN_ALLOW, t))
    
        try:
            # Get python tool from pool
            try:
                python_tool = python_pool.get(timeout=30.0)
            except queue.Empty:
                print("⚠️ Failed to get python_tool from pool, creating new one")
                python_tool = PythonTool(execution_backend="jupyter")
                try:
                    python_tool._ensure_session()
                except Exception as e:
                    print(f"⚠️ python session init failed: {e}")
                    return ""
            else:
                # Verify session is still alive
                try:
                    if python_tool._jupyter_session is None:
                        python_tool._ensure_session()
                    test_output = python_tool._jupyter_session.execute("1+1", timeout=2.0)
                    if "[ERROR]" in test_output or "Traceback" in test_output:
                        python_tool._jupyter_session = None
                        python_tool._ensure_session()
                except Exception as e:
                    print(f"⚠️ python session check failed: {e}, recreating")
                    try:
                        python_tool.close()
                    except:
                        pass
                    python_tool._jupyter_session = None
                    try:
                        python_tool._ensure_session()
                    except Exception as e2:
                        print(f"⚠️ python session recreate failed: {e2}")
                        return ""
    
            messages = self.apply_chat_template(prompt, python_tool)
            final_answer_found = ""
    
            for iteration in range(self.max_iter):
                # Check deadline
                if getattr(self, "deadline", None) and time.time() >= self.deadline:
                    print("⏰ Deadline reached")
                    break
                if final_answer_found:
                    break
    
                # Render conversation to tokens using Harmony
                prompt_ids = encoding.render_conversation_for_completion(
                    Conversation.from_messages(messages), Role.ASSISTANT
                )
                
                # Convert to tensor
                input_ids = torch.tensor([prompt_ids], dtype=torch.long)
                
                max_tokens = min(2048, self.max_model_len - len(prompt_ids))
                if max_tokens < 1:
                    print("⚠️ Context full")
                    break
    
                try:
                    # Generate with Unsloth
                    outputs = self._generate_with_unsloth(input_ids, max_new_tokens=max_tokens)
                    
                    # Extract new tokens only
                    new_token_ids = outputs[0, len(prompt_ids):].tolist()
                    token_buffer_str = self.tokenizer.decode(new_token_ids, skip_special_tokens=False)
                    
                    print(f"[Iter {iteration+1}] Generated {len(new_token_ids)} tokens")
                    
                    if len(new_token_ids) == 0:
                        break
                    
                    # Check for boxed answer
                    if "}" in token_buffer_str and self.extract_boxed_text(token_buffer_str) is not None:
                        final_answer_found = token_buffer_str
                        break
                    
                    # Parse completion into messages
                    try:
                        new_messages = encoding.parse_messages_from_completion_tokens(
                            new_token_ids, Role.ASSISTANT
                        )
                    except Exception as e:
                        print(f"Error parsing completion: {e}")
                        break
    
                    messages.extend(new_messages)
                    last_message = messages[-1]
    
                    if last_message.channel == "final" or (new_token_ids and new_token_ids[-1] == 200002):
                        break
    
                    # Handle python tool call
                    if last_message.recipient == "python":
                        if getattr(self, "deadline", None) and time.time() >= self.deadline:
                            break
    
                        py_timeout = _compute_py_timeout()
                        if py_timeout < 0.5:
                            print(f"⏰ Not enough time for python ({py_timeout:.2f}s)")
                            break
    
                        print("🐍 Executing Python code...")
                        try:
                            response_msgs = python_tool.process_sync_plus(last_message, timeout=py_timeout)
                            messages.extend(response_msgs)
                        except Exception as e:
                            print(f"⚠️ python tool failed: {e}")
                            break
    
                except Exception as e:
                    import traceback
                    print(f"⚠️ Generation error: {e}")
                    print(traceback.format_exc())
                    break
    
            if final_answer_found:
                return final_answer_found
    
            return encoding.decode_utf8(
                encoding.render_conversation_for_training(
                    Conversation.from_messages(messages),
                    RenderConversationConfig(auto_drop_analysis=False),
                )
            )
    
        except KeyboardInterrupt:
            raise
        except Exception as e:
            import traceback
            print(f"Error in generation: {e}")
            print(traceback.format_exc())
            return ""
        finally:
            # Return tool to pool
            if python_tool is not None:
                try:
                    if python_tool._jupyter_session is not None:
                        try:
                            test_output = python_tool._jupyter_session.execute("1+1", timeout=1.0)
                            if "[ERROR]" not in test_output and "Traceback" not in test_output:
                                python_pool.put(python_tool, block=False)
                            else:
                                python_tool.close()
                        except:
                            python_tool.close()
                    else:
                        python_pool.put(python_tool, block=False)
                except:
                    try:
                        python_tool.close()
                    except:
                        pass

    def _inference_sequential(self, prompts: list[str]) -> list[str]:
        """Run inference sequentially (Unsloth doesn't parallelize well on single GPU)."""
        raw_responses = []
        answers_collected = []
        majority_threshold = len(prompts) / 2
    
        print(f"🚀 Running {len(prompts)} samples sequentially...")
    
        for i, p in enumerate(prompts):
            print(f"\n--- Sample {i+1}/{len(prompts)} ---")
            
            # Check if we already have majority
            if answers_collected:
                counts = Counter(answers_collected)
                most_common_ans, count = counts.most_common(1)[0]
                if count > majority_threshold:
                    print(f"🎯 Majority reached early! {most_common_ans} appeared {count} times")
                    raw_responses.extend([""] * (len(prompts) - i))
                    break
            
            result_text = self.single_generate_tir(p, seed_offset=i)
            raw_responses.append(result_text)
            
            ans = self.extract_boxed_text(result_text)
            if ans is not None:
                answers_collected.append(ans)
                print(f"✓ Extracted answer: {ans}")
    
        return raw_responses

    def extract_boxed_text(self, text: str) -> int | None:
        """Extract a numeric answer from '\\boxed{}' or 'final answer is ...' in the text."""
        pattern = r'oxed{(.*?)}'
        matches = re.findall(pattern, str(text))
        if matches:
            for match in reversed(matches):
                if match:
                    try:
                        clean_match = match.strip().replace(',', '').replace(' ', '')
                        val = int(float(clean_match[:20]))
                        if 0 <= val <= 99999:
                            return val
                    except:
                        pass

        pattern = r'(?i)final\s+answer\s*(?:is|:)?\s*(\d+)'
        matches = re.findall(pattern, text)
        if matches:
            for match in reversed(matches):
                if match:
                    try:
                        val = int(match)
                        if 0 <= val <= 99999:
                            return val
                    except:
                        pass

        return None

    def parse_responses(self, responses: list[str]) -> int:
        """Decide on the final answer from all responses by majority vote."""
        answers = [self.extract_boxed_text(r) for r in responses]
        valid_answers = [a for a in answers if a is not None]
        
        if not valid_answers:
            print("No valid answers found")
            return 0

        counter = Counter(valid_answers)
        print(f"Answers: {counter}")

        most_common_list = counter.most_common(2)
        if len(most_common_list) > 1 and most_common_list[0][1] == most_common_list[1][1]:
            tied_answers = [ans for ans, cnt in counter.items() if cnt == most_common_list[0][1]]
            answer = max(tied_answers)
        else:
            answer = most_common_list[0][0]
        return answer

In [8]:
# Initialize the inferencer with Unsloth model
inferencer = UnslothTIRInferencer(
    model=model,
    tokenizer=tokenizer,
    k=K,
)
print("Inferencer ready!")

Inferencer ready!


# Test on Math Problem (GSM8K / MATH Dataset)

In [13]:
# Download a sample problem from GSM8K dataset (well-known math reasoning benchmark)
from datasets import load_dataset

print("Loading GSM8K dataset...")
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
print(f"Loaded {len(gsm8k)} test problems")

# Pick a sample problem (index 0)
sample_idx = 90
sample_problem = gsm8k[sample_idx]

problem_text = sample_problem["question"]
ground_truth_answer = sample_problem["answer"]

# Extract numeric answer from GSM8K format (answer after ####)
import re
gt_match = re.search(r"####\s*(-?\d+)", ground_truth_answer)
gt_numeric = int(gt_match.group(1).replace(",", "")) if gt_match else None

print("=" * 60)
print("PROBLEM:")
print(problem_text)
print("=" * 60)
print(f"GROUND TRUTH ANSWER: {gt_numeric}")
print("=" * 60)

Loading GSM8K dataset...
Loaded 1319 test problems
PROBLEM:
Ted the T-Rex was planning to bring potato salad to the dinosaur picnic.  He knows that an adult dinosaur will eat 10 lbs of potato salad, and a child will eat half as much as an adult.  If there will be 20 adults and 5 children at the picnic, how many pounds of potato salad does Ted need to bring to the picnic if he hopes to have enough to feed everyone?
GROUND TRUTH ANSWER: 225


In [14]:
# Run inference on the sample problem
print("\n🚀 Running inference...")
start_time = time.time()

predicted_answer, _ = inferencer.inference(problem_text)

elapsed = time.time() - start_time
print(f"\n⏱️ Inference took {elapsed:.2f} seconds")

# Check result
print("\n" + "=" * 60)
print("RESULTS:")
print("=" * 60)
print(f"Predicted Answer: {predicted_answer}")
print(f"Ground Truth:     {gt_numeric}")

if gt_numeric is not None:
    is_correct = (predicted_answer == gt_numeric)
    if is_correct:
        print("\n✅ CORRECT!")
    else:
        print("\n❌ INCORRECT!")
else:
    print("\n⚠️ Could not parse ground truth answer for comparison")
    
print("=" * 60)


🚀 Running inference...
Samples: 4
🚀 Running 4 samples sequentially...

--- Sample 1/4 ---
[Iter 1] Generated 210 tokens
✓ Extracted answer: 225

--- Sample 2/4 ---
[Iter 1] Generated 336 tokens
✓ Extracted answer: 225

--- Sample 3/4 ---
[Iter 1] Generated 243 tokens
✓ Extracted answer: 225

--- Sample 4/4 ---
🎯 Majority reached early! 225 appeared 3 times
[inference] Took 72.41s
Answers: Counter({225: 3})

⏱️ Inference took 72.41 seconds

RESULTS:
Predicted Answer: 225
Ground Truth:     225

✅ CORRECT!
